# Lab 06 — Snowpark development, deployment and orchestration

Copyright © 2026 Innovation In Software Corporation. All rights reserved.

This is the single required Lab 06 notebook: SQL setup → Python development → reversible trial → SQL procedure deployment → task orchestration and checks. Complete Labs 01–05 first (10 raw/inbox deliveries, 3 current orders, active value USD 165, empty stream).

Import this notebook into Snowflake with role **ADV_LAB**, warehouse **ADV_WH**, and an available warehouse Python runtime. Add `snowflake-snowpark-python` without a version pin. The deployed procedure uses Python 3.12. Execute one cell at a time with the instructor. **Do not Run All or schedule this development notebook:** it contains rollback trials and asynchronous checkpoints.

SQL cells use ADV_ORDERS and schema OPS; Python transformations explicitly reference SILVER and BRONZE. No local Python setup is required. Notebook hosting still needs rehearsal in the student environment.


In [ ]:
USE ROLE ADV_LAB;
USE WAREHOUSE ADV_WH;
USE DATABASE ADV_ORDERS;
USE SCHEMA OPS;

## A. Establish destinations and reference data

Create the known target tables first; preserve existing data with IF NOT EXISTS. Inspect available package versions before developing the transformation.


In [ ]:
-- Use a common session time zone for classroom checks and LTZ history display.
-- This is session setup, not a conversion of stored event timestamps.
-- Fixture event_time strings already contain Z (UTC), so their TIMESTAMP_TZ
-- interpretation does not depend on this setting. Timestamps without an offset
-- would use the session zone when converted to TIMESTAMP_TZ.

ALTER SESSION SET TIMEZONE = 'UTC';

In [ ]:
-- Label statements in this session so we can retrieve them at the end.

ALTER SESSION SET QUERY_TAG = 'ADV_2026_DAY1';

In [ ]:
-- ──────────────────────────────────────────────────────────────────────────────
-- PART A: SETUP - prepare destinations before developing transformations
-- ──────────────────────────────────────────────────────────────────────────────

-- The first five files establish 3 current orders and active value USD 165.
-- SQL is a useful baseline; Snowpark implements the same contract.
-- Python runs inside Snowflake, with the same setup for every student.

CREATE TABLE IF NOT EXISTS SILVER.REGION_REFERENCE
(
    REGION_CODE  STRING,
    REGION_LABEL STRING
);

In [ ]:
CREATE TABLE IF NOT EXISTS SILVER.SNOWPARK_EVENTS
(
    EVENT_ID          STRING,
    ORDER_ID          STRING,
    VERSION           NUMBER(3, 1),
    EVENT_TIME        TIMESTAMP_TZ,
    STATUS            STRING,
    AMOUNT            NUMBER(12, 2),
    REGION            STRING,
    REGION_LABEL      STRING,
    ACTIVE_AMOUNT_USD NUMBER(12, 2),
    REJECT_REASON     STRING,
    SOURCE_FILE       STRING,
    SOURCE_ROW        NUMBER
);

In [ ]:
CREATE TABLE IF NOT EXISTS SILVER.SNOWPARK_CANDIDATES LIKE SILVER.ORDERS_CURRENT;

In [ ]:
-- The three table definitions above establish the destinations before processing.
-- REGION_REFERENCE enriches region labels; SNOWPARK_EVENTS holds classified
-- deliveries; SNOWPARK_CANDIDATES holds one selected state per order.
-- The handler joins, populates these tables and merges into existing ORDERS_CURRENT.

MERGE INTO SILVER.REGION_REFERENCE t
USING (SELECT column1 AS region_code, column2 AS region_label
       FROM VALUES ('US', 'United States'), ('EU', 'Europe')) s
ON t.region_code = s.region_code
WHEN MATCHED THEN UPDATE SET region_label = s.region_label
WHEN NOT MATCHED THEN INSERT (region_code, region_label)
VALUES (s.region_code, s.region_label);

In [ ]:
-- Query package versions available in this account; do not assume a fixed release.
-- Package version is omitted in deployment: resolve latest available for Python 3.12.

SELECT DISTINCT version
FROM SNOWFLAKE.INFORMATION_SCHEMA.PACKAGES
WHERE language = 'python'
  AND package_name = 'snowflake-snowpark-python'
ORDER BY TRY_TO_NUMBER(SPLIT_PART(version, '.', 1)) DESC,
         TRY_TO_NUMBER(SPLIT_PART(version, '.', 2)) DESC,
         TRY_TO_NUMBER(SPLIT_PART(version, '.', 3)) DESC
LIMIT 10;

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark
session = get_active_session()


In [ ]:
role = session.sql("SELECT CURRENT_ROLE()").collect()[0][0]
assert role == "ADV_LAB", "Use the student ADV_LAB role"
session.use_database("ADV_ORDERS")
session.use_schema("OPS")
session.use_warehouse("ADV_WH")


In [ ]:
# Common session zone for history/LTZ display; fixture event times already include Z.

session.sql("ALTER SESSION SET TIMEZONE = 'UTC'").collect()
# Separate notebook session: set its own tag for the SQL retrieval example.

session.sql("ALTER SESSION SET QUERY_TAG = 'ADV_2026_DAY1'").collect()
print("Notebook Snowpark version:", snowflake.snowpark.__version__)


## 1. Read existing deliveries
`session.table` creates a DataFrame plan. `.show()` executes the plan for inspection; `.count()` executes an aggregate. Assignment alone does not load all events into Python memory. Expect inbox=10 on the first pass, or 11 after completing the replay. Current orders=3 and the stream must be empty before this development exercise.

In [ ]:
"""Reusable Snowpark transformations for the Day 1 order-event pipeline.
Copyright © 2026 Innovation In Software Corporation. All rights reserved.
"""
from snowflake.snowpark.functions import (
    col, sql_expr, row_number, when_matched, when_not_matched
)
from snowflake.snowpark.window import Window

ORDER_COLUMNS = ["ORDER_ID", "EVENT_ID", "VERSION", "EVENT_TIME", "STATUS", "AMOUNT", "REGION"]
EVENT_COLUMNS = ["EVENT_ID", "ORDER_ID", "VERSION", "EVENT_TIME", "STATUS", "AMOUNT",
                 "REGION", "REGION_LABEL", "ACTIVE_AMOUNT_USD", "REJECT_REASON",
                 "SOURCE_FILE", "SOURCE_ROW"]

raw = session.table("SILVER.EVENT_INBOX")
assert raw.count() in (10, 11), "Complete Labs 01-05 before this notebook"
assert session.table("BRONZE.EVENTS_STREAM").count() == 0, "Finish processing pending events first"
raw.select("PAYLOAD", "SOURCE_FILE", "SOURCE_ROW").show()


## 2. Convert to typed columns
Define one reusable transformation, call it, then inspect its schema and rows. e3 amount becomes NULL. For this classroom example, VERSION uses NUMBER(3,1). Valid revisions are whole numbers; 1.5 still fails validation. This limited precision is a teaching simplification, not a general validator for arbitrary source versions. `select_expr` uses SQL expressions inside a Snowpark DataFrame, without collecting the event data.

In [ ]:
def type_events(df):
    # select_expr builds a DataFrame plan; it does not collect the event rows.
    # Classroom versions use NUMBER(3,1) to keep the displayed values compact.

    return df.select_expr(
        "NULLIF(TRIM(payload:event_id::STRING), '') AS EVENT_ID",
        "NULLIF(TRIM(payload:order_id::STRING), '') AS ORDER_ID",
        "TRY_TO_DECIMAL(payload:version::STRING, 3, 1) AS VERSION",
        "TRY_TO_TIMESTAMP_TZ(payload:event_time::STRING) AS EVENT_TIME",
        "payload:status::STRING AS STATUS",
        "TRY_TO_DECIMAL(payload:amount::STRING, 12, 2) AS AMOUNT",
        "payload:region::STRING AS REGION",
        "SOURCE_FILE", "SOURCE_ROW")

typed = type_events(raw)
print(typed.schema)
typed.select("EVENT_ID", "ORDER_ID", "VERSION", "AMOUNT").sort("EVENT_ID").show()


## 3. Apply the business validation rules
`with_column` adds a classification expression to the plan. The CASE rules match Lab 03. Expect exactly e3 / BAD_AMOUNT; bad data stays in the inbox.

In [ ]:
def classify_events(df):
    # Reuse the business rules developed in SQL; express a new DataFrame column.

    return df.with_column("REJECT_REASON",
                          sql_expr("""CASE
                                        WHEN event_id IS NULL OR order_id IS NULL THEN 'MISSING_KEY'
                                        WHEN version IS NULL OR version < 1 OR version <> TRUNC(version) THEN 'BAD_VERSION'
                                        WHEN event_time IS NULL THEN 'BAD_TIME'
                                        WHEN status IS NULL OR status NOT IN ('PLACED', 'PAID', 'CANCELLED') THEN 'BAD_STATUS'
                                        WHEN amount IS NULL OR amount < 0 THEN 'BAD_AMOUNT'
                                        WHEN region IS NULL OR region NOT IN ('US', 'EU') THEN 'BAD_REGION'
                                        ELSE NULL
                                    END"""))

classified = classify_events(typed)
rejected = classified.filter(col("REJECT_REASON").is_not_null())
rejected.select("EVENT_ID", "REJECT_REASON").show()
assert rejected.count() == 1


## 4. Enrich with a reference join
Join the region labels and derive active amount. Cancellation has active amount zero. This is a left join so missing reference data would not silently remove an event. Inspect both the reference and the enriched result.

In [ ]:
def enrich_events(df_classified, df_reference):
    enriched = df_classified.join(
        df_reference, df_classified["REGION"] == df_reference["REGION_CODE"], "left")
    return enriched.with_column("ACTIVE_AMOUNT_USD", sql_expr("IFF(status = 'CANCELLED', 0, amount)"))

reference = session.table("SILVER.REGION_REFERENCE")
reference.show()

enriched = enrich_events(classified, reference)
enriched.select("EVENT_ID", "REGION", "REGION_LABEL", "STATUS", "ACTIVE_AMOUNT_USD").sort("EVENT_ID").show()


## 5. Check conflicts and select one latest state per order
Stop if one immutable event ID has conflicting valid payloads. The existing SQL conflict view has the same classroom limitation noted in Lab 03: TO_JSON is not a canonical JSON comparison.

The first window selects a representative delivery per event ID, keeping delivery lineage for inspection. The second window selects the latest business state per order. Neither window deletes inbox rows. Expect o1/e4/120, o2/e5/CANCELLED/50 and o3/e8/45.

In [ ]:
def select_candidates(df):
    good = df.filter(col("REJECT_REASON").is_null())

    # Pick a representative delivery of each immutable event; keep inbox history.
    # File/row identifies delivery lineage, not business-state precedence.

    unique = good.with_column("RN", row_number().over(Window.partition_by("EVENT_ID").order_by(col("SOURCE_FILE"), col("SOURCE_ROW"))))
    unique = unique.filter(col("RN") == 1).drop("RN")

    # Now choose the latest business state using the same SQL precedence rule.

    latest = unique.with_column("RN", row_number().over(Window.partition_by("ORDER_ID").order_by(col("VERSION").desc(), col("EVENT_TIME").desc(), col("EVENT_ID").desc())))
    return latest.filter(col("RN") == 1).select(*ORDER_COLUMNS)

assert session.table("SILVER.EVENT_CONFLICTS").count() == 0, "Resolve event-ID conflicts before continuing"

candidates = select_candidates(enriched)
candidates.sort("ORDER_ID").show()

assert candidates.count() == 3


## 6. Compare with the established SQL result
Show the matching rows through DataFrame `intersect`. Also check total counts and both differences: matches alone would hide extra rows. This develops transformations without changing the accepted current state.

In [ ]:
baseline = session.table("SILVER.ORDERS_CURRENT").select(*ORDER_COLUMNS)

candidates.intersect(baseline).sort("ORDER_ID").show()

assert baseline.count() == 3

assert candidates.subtract(baseline).count() == 0

assert baseline.subtract(candidates).count() == 0


## 7. Express MERGE through Snowpark
`Table.merge` uses `when_matched(condition)` and `when_not_matched()` clauses. Matching identifies the order; the additional condition tests whether its state is newer. Define the function first. Definition alone does not execute MERGE.

In [ ]:
def merge_current(sf_session, df_candidates):
    target = sf_session.table("SILVER.ORDERS_CURRENT")
    newer = (
        (df_candidates["VERSION"] > target["VERSION"])
        | ((df_candidates["VERSION"] == target["VERSION"])
           & (df_candidates["EVENT_TIME"] > target["EVENT_TIME"]))
        | ((df_candidates["VERSION"] == target["VERSION"])
           & (df_candidates["EVENT_TIME"] == target["EVENT_TIME"])
           & (df_candidates["EVENT_ID"] > target["EVENT_ID"]))
    )
    updates = {name: df_candidates[name] for name in ORDER_COLUMNS if name != "ORDER_ID"}
    inserts = {name: df_candidates[name] for name in ORDER_COLUMNS}

    # Table.merge is an action: Snowflake executes MERGE and returns row counts.

    return target.merge(df_candidates, target["ORDER_ID"] == df_candidates["ORDER_ID"], [
        when_matched(newer).update(updates),
        when_not_matched().insert(inserts)
    ])


## 8. Try the writes as one reversible unit
Both scratch destinations were created in the SQL setup section of this notebook. Record their baseline counts. Inside the trial, replace their derived rows using DML, then call the merge function. The current states already agree, so expect zero inserted and updated orders. `finally` always rolls back, even if an inspection fails. Run this complete cell together; never insert DDL into this transaction.

In [ ]:
before_events = session.table("SILVER.SNOWPARK_EVENTS").count()
before_candidates = session.table("SILVER.SNOWPARK_CANDIDATES").count()

session.sql("BEGIN TRANSACTION").collect()

try:
    session.table("SILVER.SNOWPARK_EVENTS").delete()
    enriched.select(*EVENT_COLUMNS).write.insert_into("SILVER.SNOWPARK_EVENTS")

    session.table("SILVER.SNOWPARK_CANDIDATES").delete()
    candidates.write.insert_into("SILVER.SNOWPARK_CANDIDATES")

    result = merge_current(session, session.table("SILVER.SNOWPARK_CANDIDATES"))
    print(result)

    session.table("SILVER.SNOWPARK_CANDIDATES").sort("ORDER_ID").show()
finally:
    session.sql("ROLLBACK").collect()

assert session.table("SILVER.SNOWPARK_EVENTS").count() == before_events
assert session.table("SILVER.SNOWPARK_CANDIDATES").count() == before_candidates
assert session.table("BRONZE.EVENTS_STREAM").count() == 0


## 9. Assemble the verified functions into a handler
Now add transaction control, one stream-to-inbox INSERT, and the conflict guard. The transformation functions are the ones already developed above. Unlike the trial, a successful handler commits; an exception rolls back.

Define `main` below but do not call it directly in the development section. `Snowpark-OrderTransform.py` is the complete reusable source. The following SQL cells in this notebook deploy identical source inside a Python 3.12 procedure with an unpinned Snowpark dependency, calls it, then switches the existing task graph. SQL is retained for transaction control and stream capture; transformation, scratch-table DML and MERGE use Snowpark APIs.

In [ ]:
def main(sf_session):
    # Only SQL transaction control and stream capture remain sf_session.sql calls.
    # All destinations already exist; no DDL belongs inside this transaction.

    sf_session.sql("BEGIN TRANSACTION").collect()
    try:
        # Consume this stream into the inbox ONCE per transaction.

        sf_session.sql("""INSERT INTO SILVER.EVENT_INBOX
            SELECT payload, source_file, source_row, scan_time
            FROM BRONZE.EVENTS_STREAM""").collect()
        if sf_session.table("SILVER.EVENT_CONFLICTS").count():
            raise ValueError("Conflicting payloads for immutable event_id")

        typed = type_events(sf_session.table("SILVER.EVENT_INBOX"))
        classified = classify_events(typed)
        enriched = enrich_events(classified, sf_session.table("SILVER.REGION_REFERENCE"))
        candidates = select_candidates(enriched)

        sf_session.table("SILVER.SNOWPARK_EVENTS").delete()
        enriched.select(*EVENT_COLUMNS).write.insert_into("SILVER.SNOWPARK_EVENTS")
        sf_session.table("SILVER.SNOWPARK_CANDIDATES").delete()
        candidates.write.insert_into("SILVER.SNOWPARK_CANDIDATES")
        result = merge_current(sf_session, sf_session.table("SILVER.SNOWPARK_CANDIDATES"))
        sf_session.sql("COMMIT").collect()
        return f"Applied with Snowpark: {result}"
    except Exception:
        sf_session.sql("ROLLBACK").collect()
        raise


## B. Deploy the verified functions

The next SQL cell embeds the same handler just developed. CREATE PROCEDURE establishes its resolved package environment; CALL commits processing. Then reuse the existing task graph.


In [ ]:
-- ============================================================================
-- PART B: DEPLOY - package the verified functions as a stored procedure.
-- ============================================================================
-- Snowpark-OrderTransform.py and the notebook contain this same reusable source.
-- Unpinned packages resolve at CREATE time; calls reuse that resolved environment.
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-procedure

CREATE OR REPLACE PROCEDURE SILVER.APPLY_EVENTS_SNOWPARK()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.12'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
EXECUTE AS OWNER
AS
$$
"""Reusable Snowpark transformations for the Day 1 order-event pipeline.
Copyright © 2026 Innovation In Software Corporation. All rights reserved.
"""
from snowflake.snowpark.functions import (
    col, sql_expr, row_number, when_matched, when_not_matched
)
from snowflake.snowpark.window import Window

ORDER_COLUMNS = ["ORDER_ID", "EVENT_ID", "VERSION", "EVENT_TIME", "STATUS", "AMOUNT", "REGION"]
EVENT_COLUMNS = ["EVENT_ID", "ORDER_ID", "VERSION", "EVENT_TIME", "STATUS", "AMOUNT",
                 "REGION", "REGION_LABEL", "ACTIVE_AMOUNT_USD", "REJECT_REASON",
                 "SOURCE_FILE", "SOURCE_ROW"]

def type_events(df):
    # select_expr builds a DataFrame plan; it does not collect the event rows.
    # Classroom versions use NUMBER(3,1) to keep the displayed values compact.

    return df.select_expr(
        "NULLIF(TRIM(payload:event_id::STRING), '') AS EVENT_ID",
        "NULLIF(TRIM(payload:order_id::STRING), '') AS ORDER_ID",
        "TRY_TO_DECIMAL(payload:version::STRING, 3, 1) AS VERSION",
        "TRY_TO_TIMESTAMP_TZ(payload:event_time::STRING) AS EVENT_TIME",
        "payload:status::STRING AS STATUS",
        "TRY_TO_DECIMAL(payload:amount::STRING, 12, 2) AS AMOUNT",
        "payload:region::STRING AS REGION",
        "SOURCE_FILE", "SOURCE_ROW")

def classify_events(df):
    # Reuse the business rules developed in SQL; express a new DataFrame column.

    return df.with_column("REJECT_REASON",
                          sql_expr("""CASE
                                        WHEN event_id IS NULL OR order_id IS NULL THEN 'MISSING_KEY'
                                        WHEN version IS NULL OR version < 1 OR version <> TRUNC(version) THEN 'BAD_VERSION'
                                        WHEN event_time IS NULL THEN 'BAD_TIME'
                                        WHEN status IS NULL OR status NOT IN ('PLACED', 'PAID', 'CANCELLED') THEN 'BAD_STATUS'
                                        WHEN amount IS NULL OR amount < 0 THEN 'BAD_AMOUNT'
                                        WHEN region IS NULL OR region NOT IN ('US', 'EU') THEN 'BAD_REGION'
                                        ELSE NULL
                                    END"""))

def enrich_events(df_classified, df_reference):
    enriched = df_classified.join(
        df_reference, df_classified["REGION"] == df_reference["REGION_CODE"], "left")
    return enriched.with_column("ACTIVE_AMOUNT_USD", sql_expr("IFF(status = 'CANCELLED', 0, amount)"))

def select_candidates(df):
    good = df.filter(col("REJECT_REASON").is_null())

    # Pick a representative delivery of each immutable event; keep inbox history.
    # File/row identifies delivery lineage, not business-state precedence.

    unique = good.with_column("RN", row_number().over(Window.partition_by("EVENT_ID").order_by(col("SOURCE_FILE"), col("SOURCE_ROW"))))
    unique = unique.filter(col("RN") == 1).drop("RN")

    # Now choose the latest business state using the same SQL precedence rule.

    latest = unique.with_column("RN", row_number().over(Window.partition_by("ORDER_ID").order_by(col("VERSION").desc(), col("EVENT_TIME").desc(), col("EVENT_ID").desc())))
    return latest.filter(col("RN") == 1).select(*ORDER_COLUMNS)

def merge_current(sf_session, df_candidates):
    target = sf_session.table("SILVER.ORDERS_CURRENT")
    newer = (
        (df_candidates["VERSION"] > target["VERSION"])
        | ((df_candidates["VERSION"] == target["VERSION"])
           & (df_candidates["EVENT_TIME"] > target["EVENT_TIME"]))
        | ((df_candidates["VERSION"] == target["VERSION"])
           & (df_candidates["EVENT_TIME"] == target["EVENT_TIME"])
           & (df_candidates["EVENT_ID"] > target["EVENT_ID"]))
    )
    updates = {name: df_candidates[name] for name in ORDER_COLUMNS if name != "ORDER_ID"}
    inserts = {name: df_candidates[name] for name in ORDER_COLUMNS}

    # Table.merge is an action: Snowflake executes MERGE and returns row counts.

    return target.merge(df_candidates, target["ORDER_ID"] == df_candidates["ORDER_ID"], [
        when_matched(newer).update(updates),
        when_not_matched().insert(inserts)
    ])

def main(sf_session):
    # Only SQL transaction control and stream capture remain sf_session.sql calls.
    # All destinations already exist; no DDL belongs inside this transaction.

    sf_session.sql("BEGIN TRANSACTION").collect()
    try:
        # Consume this stream into the inbox ONCE per transaction.

        sf_session.sql("""INSERT INTO SILVER.EVENT_INBOX
            SELECT payload, source_file, source_row, scan_time
            FROM BRONZE.EVENTS_STREAM""").collect()
        if sf_session.table("SILVER.EVENT_CONFLICTS").count():
            raise ValueError("Conflicting payloads for immutable event_id")

        typed = type_events(sf_session.table("SILVER.EVENT_INBOX"))
        classified = classify_events(typed)
        enriched = enrich_events(classified, sf_session.table("SILVER.REGION_REFERENCE"))
        candidates = select_candidates(enriched)

        sf_session.table("SILVER.SNOWPARK_EVENTS").delete()
        enriched.select(*EVENT_COLUMNS).write.insert_into("SILVER.SNOWPARK_EVENTS")
        sf_session.table("SILVER.SNOWPARK_CANDIDATES").delete()
        candidates.write.insert_into("SILVER.SNOWPARK_CANDIDATES")
        result = merge_current(sf_session, sf_session.table("SILVER.SNOWPARK_CANDIDATES"))
        sf_session.sql("COMMIT").collect()
        return f"Applied with Snowpark: {result}"
    except Exception:
        sf_session.sql("ROLLBACK").collect()
        raise
$$;

In [ ]:
DESC PROCEDURE SILVER.APPLY_EVENTS_SNOWPARK();

In [ ]:
-- Inspect the resolved package dependency in the procedure description.

CALL SILVER.APPLY_EVENTS_SNOWPARK();

In [ ]:
SELECT *
FROM SILVER.SNOWPARK_CANDIDATES
ORDER BY ORDER_ID;

In [ ]:
SELECT EVENT_ID, REJECT_REASON
FROM SILVER.SNOWPARK_EVENTS
WHERE REJECT_REASON IS NOT NULL;

In [ ]:
-- CHECKPOINT: candidates o1/e4/120, o2/e5/50 cancelled, o3/e8/45; one BAD_AMOUNT.

SELECT ORDER_ID, EVENT_ID, VERSION, EVENT_TIME, STATUS, AMOUNT, REGION
FROM SILVER.SNOWPARK_CANDIDATES
MINUS
SELECT ORDER_ID, EVENT_ID, VERSION, EVENT_TIME, STATUS, AMOUNT, REGION
FROM SILVER.ORDERS_CURRENT;

In [ ]:
SELECT ORDER_ID, EVENT_ID, VERSION, EVENT_TIME, STATUS, AMOUNT, REGION
FROM SILVER.ORDERS_CURRENT
MINUS
SELECT ORDER_ID, EVENT_ID, VERSION, EVENT_TIME, STATUS, AMOUNT, REGION
FROM SILVER.SNOWPARK_CANDIDATES;

In [ ]:
-- CHECKPOINT: both differences are empty. Equality does not prove all contracts;
-- inspect rejected rows and the source replay next.

-- ──────────────────────────────────────────────────────────────────────────────
-- PART C: ORCHESTRATE - switch the existing task graph to the procedure
-- ──────────────────────────────────────────────────────────────────────────────

-- If 99-Stop was run earlier, this also restarts the child. Root stays suspended.

ALTER TASK OPS.APPLY_TASK SUSPEND;

In [ ]:
ALTER TASK OPS.AUDIT_TASK SUSPEND;

In [ ]:
-- Wait for any in-flight run to finish before modifying the graph.

SHOW TASKS IN SCHEMA OPS;

## Pause: verify the task graph is idle

After suspending both tasks, wait for any in-flight run to finish. SHOW TASKS shows suspension state; use TASK_HISTORY below to check running executions. Do not modify the graph until no execution is running.


In [ ]:
SELECT NAME, STATE, QUERY_ID, ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(RESULT_LIMIT => 100))
WHERE DATABASE_NAME = 'ADV_ORDERS'
  AND SCHEMA_NAME = 'OPS'
ORDER BY SCHEDULED_TIME DESC;

In [ ]:
ALTER TASK OPS.APPLY_TASK MODIFY AS CALL ADV_ORDERS.SILVER.APPLY_EVENTS_SNOWPARK();

In [ ]:
ALTER TASK OPS.AUDIT_TASK RESUME;

In [ ]:
-- Add one duplicate delivery in a distinct synthetic file lineage.
-- The NOT EXISTS guard makes this replay step rerunnable without adding more rows.
-- It intentionally bypasses file ingestion to isolate business replay behavior.

INSERT INTO BRONZE.EVENTS_RAW
SELECT PAYLOAD, 'snowpark/replay-e8', 1, CURRENT_TIMESTAMP()
FROM BRONZE.EVENTS_RAW
WHERE PAYLOAD:event_id::STRING = 'e8'
  AND NOT EXISTS
      (SELECT 1 FROM BRONZE.EVENTS_RAW WHERE SOURCE_FILE = 'snowpark/replay-e8')
QUALIFY ROW_NUMBER() OVER (ORDER BY SOURCE_FILE, SOURCE_ROW) = 1;

In [ ]:
EXECUTE TASK OPS.APPLY_TASK;

In [ ]:
SELECT NAME, STATE, QUERY_ID, ERROR_MESSAGE, COMPLETED_TIME
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(RESULT_LIMIT => 100))
WHERE DATABASE_NAME = 'ADV_ORDERS'
  AND SCHEMA_NAME = 'OPS'
ORDER BY SCHEDULED_TIME DESC;

## Pause: wait for asynchronous completion

EXECUTE TASK submits work; it does not wait for the graph. Repeat the preceding history cell until APPLY_TASK and AUDIT_TASK succeed, and inspect errors if they fail. On rerun, the guarded replay inserts zero rows; with an empty stream the root can be SKIPPED and the child does not run. That is expected. Do not check final data while an execution is still running.


In [ ]:
-- First pass: wait for both tasks to succeed. Rerun with an empty stream may skip.

SELECT COUNT(*) AS PENDING_ROWS
FROM BRONZE.EVENTS_STREAM;

In [ ]:
SELECT *
FROM GOLD.ORDER_METRICS;

In [ ]:
-- CHECKPOINT after success: stream 0; raw/inbox 11; current 3; active value 165.
-- Preserve cancellation and late-version protections. Repeating the handler
-- changes neither current order state nor event counts when stream is empty.

CALL SILVER.APPLY_EVENTS_SNOWPARK();

In [ ]:
-- STEP: retrieve this student's tagged statements from the past hour.
-- Includes worksheet and notebook sessions for the current user; no account-wide
-- history privilege is needed for the student's own statements.
-- The 1000-result limit applies BEFORE the tag filter: narrow the time range
-- if recent activity would push the desired statements outside that window.
-- QUERY_TAG is session-scoped. A task uses its own session and does not inherit
-- this worksheet tag; use TASK_HISTORY above to verify the task graph.
-- Source: https://docs.snowflake.com/en/sql-reference/functions/query_history

SELECT query_id, start_time, execution_status, total_elapsed_time, query_text
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER(
    END_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 1000,
    INCLUDE_CLIENT_GENERATED_STATEMENT => TRUE))
WHERE query_tag = 'ADV_2026_DAY1'
ORDER BY start_time DESC;

## Finish Day 1

Required: run **99-Stop.sql**, verify the final object states, and end this notebook session using its controls. Closing the browser does not stop background work. The optional Snowpark-Pipeline.ipynb / Optional-NotebookPipeline.sql branch requires separate hosting validation and must never run concurrently with the procedure consumer.
